In [ ]:
"""
Author: Sophie A. Liu
Date: 05/28/2026 5:42pm
Purpose: NSF to isolate genes. based on Townes & Engelhardt (2022) but updated jupyter-- theirs is outdated
"""

In [1]:
import squidpy as sq                      # vers. 1.11.0 of scanpy

import torch                              # tensorflow is incompatible with python 3.14
import gpytorch
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

c:\Users\saliu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# working directory
import os
os.chdir("i:/Hu Lab/Sophie/1. Cell death/PD1-9_realign/outs/binned_outputs")

In [3]:
# loading in data/ initializing
np.random.seed(42)                        # the answer to life, the universe, and everything
torch.manual_seed(42)                     # no, I will never tire of putting this in my code
                  
eps = 1e-8                                # numerical stability so no log(0)

pd1_vis = sq.read.visium("square_008um", load_images=True)

c:\Users\saliu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [16]:
from sklearn.cluster import KMeans
X = pd1_vis.obsm["spatial"]               # spatial coordinates
Y = pd1_vis.X.toarray()                   # gene expression coerced into dense matrix   

Y = Y[:, :2000]                           # if I don't reduce this I will crash the computer. they reduced to 80
sz = Y.sum(axis = 1)                      # total RNA counts at each coordinate

Z = KMeans(n_clusters=500).fit(X).cluster_centers_   # 500 chosen from their paper

In [17]:
# must convert for math
X = torch.tensor(X, dtype = torch.float32)
Y = torch.tensor(Y, dtype = torch.float32)
sz = torch.tensor(sz, dtype = torch.float32)

Z = torch.tensor(Z, dtype = torch.float32)

In [6]:
# Matern kernel 3/2 is used in their paper. have to define explicitly for torch
def matern32(dist):
    sqrt3 = 1.732
    return (1 + sqrt3 * dist) * torch.exp(-sqrt3 * dist)

In [11]:
# taking spatial coords and smoothing the gene expressions
class SpatialFactor(torch.nn.Module):
    def __init__(self, Z):
        super().__init__()
        self.Z = Z
        self.u = torch.nn.Parameter(torch.randn(Z.shape[0]))                # learnable weights

    def forward(self, x):                                                   # computing factors
        dist = torch.cdist(x, self.Z)
        K_xz = matern32(dist)                                            # converting distance to similarity
        return K_xz @ self.u

In [12]:
# deriving the factors
class NSF(torch.nn.Module):
    def __init__(self, X, Y, Z, L=7):                                       # for now lucky number 7 factors
        super().__init__()
        self.L = L
        self.Y = Y

        self.factors = torch.nn.ModuleList([
            SpatialFactor(Z) for _ in range(L)
        ])

        self.W = torch.nn.Parameter(torch.randn(L, Y.shape[1]) * 0.01)     # which genes belong to each factor

    def forward(self, x, sz):
        F = torch.stack([f(x) for f in self.factors], dim=1)               # spatial activity (cells x num factors)
        log_mu = F @ self.W + torch.log(sz.unsqueeze(-1) + eps)            # combining factors into genes
                                                                           # while correcting for sequencing depth like their paper
        return log_mu, F

In [13]:
# seeing how well the predicted expression matches the actual data
def nb_loglik(y, mu, theta=1.0):                                           # noise level 1.0 for dispersion
    theta = torch.tensor(theta, dtype=y.dtype, device=y.device)

    ll = torch.lgamma(y + theta) - torch.lgamma(theta) - torch.lgamma(y + 1)

    p = theta / (theta + mu + eps)                                         # probability form

    ll += theta * torch.log(p + eps)
    ll += y * torch.log(1 - p + eps)

    return ll.sum()                                                        # total score over all bins

In [ ]:
# the actual training part
model = NSF(X, Y, Z, L=7)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(20):                                                    # epoch complete passes. for now small. plot losses to see plateau later
    log_mu, F = model(X, sz)
    mu = torch.exp(log_mu)

    loss = -nb_loglik(Y, mu)                                               # want to minimize negative likelihood

    opt.zero_grad()                                                        # resetting old gradients for new
    loss.backward()
    opt.step()

In [ ]:
W = model.W.detach().cpu().numpy()
genes = np.array(pd1_vis.var_names)

sorted = np.argsort(W[3, :])[::-1][:40]                                    # for factor 3. appending weights model.W[3]
top_genes = genes[sorted]

In [20]:
print(top_genes)

['Agpat2' 'Paqr8' 'Dgkz' 'Tram1' 'Zbtb41' 'Mcm3' 'Tor4a' 'Swi5' 'Olfr1408'
 'Serpinb3d' 'Kynu' 'Fjx1' 'Traf3ip1' 'Cel' 'Sema4c' 'Ifi208' 'Rnf152'
 'Ccnyl1' 'Kif28' 'Glul' 'D1Pas1' 'Lmbrd1' 'Taf1a' 'Serpinb8' 'Sept2'
 'Coq4' 'Spopl' 'Echdc3' 'Lhx4' 'Olfr1118' 'Skida1' 'Rab3gap2' 'Cacnb4'
 'Flvcr1' 'Kansl3' 'Lypd1' 'Evx2' 'Brinp2' 'Mybph' 'Nek6']
